이 자료는 위키독스 딥 러닝을 이용한 자연어 처리 입문의 TF-IDF 튜토리얼 자료입니다.  

링크 : https://wikidocs.net/31698

# 1. 문서 단어 행렬(Document-Term Matrix, DTM)
- 각 문서에 대한 BoW를 하나의 행렬로 만든 것
- 서로 다른 문서들을 비교하기 위해 사용

# 2. 문서 단어 행렬의 한계
## 2.1 희소 표현(Sparse representation)
- 희소 벡터는 많은 양의 저장 공간과 높은 계산 복잡도를 요구
- DTM의 각 문서 벡터의 차원은 전체 단어 집합의 크기를 가지며, 대부분의 값이 0을 가질 수도 있음
- 전처리를 통해 단어 집합의 크기를 줄이는 것이 중요

## 2.2 단순 빈도 수 기반 접근
- 각 문서에는 중요한 단어와 불필요한 단어들이 혼재되어 있음
- 관사, 조사 등의 문법적 요소는 문서의 유사성과 무차별하게 모든 문서에서 많이 등장할 수 있음

# 3. TF-IDF(단어 빈도-역 문서 빈도)
- 단어의 빈도와 역 문서 빈도(문서의 빈도에 특정 식을 취함)를 사용하여 가중치로 주는 방법
- 우선 DTM을 만든 후, TF-IDF 가중치를 부여
- 주로 문서의 유사도를 구하는 작업, 문서 내에서 특정 단어의 중요도를 구하는 작업 등에 활용

## 3.1 $\text{tf}(d,t)$
- 특정 문서 d에서의 특정 단어 t의 등장 횟수
- DTM에서 각 단어들이 가진 값

## 3.2 $\text{df}(t)$
- 특정 단어 t가 등장한 문서의 수
- DTM에서 각 단어가 등장하는 행 수

## 3.3 $\text{idf}(t)$
- df(t)의 역수에 $\log$를 취하고, 분모가 0이 되는 것을 방지하기 위해 1을 더한 값
$$ \text{idf}(t) = \ln(\frac{n}{1+\text{df}(t)}) $$

# 4. 파이썬으로 TF-IDF 직접 구현하기

In [1]:
from math import log
import pandas as pd
 
docs = [
  '먹고 싶은 사과',
  '먹고 싶은 바나나',
  '길고 노란 바나나 바나나',
  '저는 과일이 좋아요'
] 
 
vocab = list(set(w for doc in docs for w in doc.split()))
vocab.sort()
print('단어장의 크기 :', len(vocab))
print(vocab)

단어장의 크기 : 9
['과일이', '길고', '노란', '먹고', '바나나', '사과', '싶은', '저는', '좋아요']


In [2]:
# 총 문서의 수
N = len(docs) 
 
def tf(t, d):
  return d.count(t)
 
def idf(t):
  df = 0
  for doc in docs:
    df += t in doc
  return log(N/(df+1))
 
def tfidf(t, d):
  return tf(t,d)* idf(t)

In [3]:
result = []

# 각 문서에 대해서 아래 연산을 반복
for i in range(N):
  result.append([])
  d = docs[i]
  for j in range(len(vocab)):
    t = vocab[j]
    result[-1].append(tf(t, d))
        
tf_ = pd.DataFrame(result, columns = vocab)

In [4]:
tf_

,과일이,길고,노란,먹고,바나나,사과,싶은,저는,좋아요
0,0,0,0,1,0,1,1,0,0
1,0,0,0,1,1,0,1,0,0
2,0,1,1,0,2,0,0,0,0
3,1,0,0,0,0,0,0,1,1


In [5]:
result = []
for j in range(len(vocab)):
    t = vocab[j]
    result.append(idf(t))

idf_ = pd.DataFrame(result, index=vocab, columns=["IDF"])
idf_

,IDF
과일이,0.693147
길고,0.693147
노란,0.693147
먹고,0.287682
바나나,0.287682
사과,0.693147
싶은,0.287682
저는,0.693147
좋아요,0.693147


In [6]:
result = []
for i in range(N):
  result.append([])
  d = docs[i]
  for j in range(len(vocab)):
    t = vocab[j]
    result[-1].append(tfidf(t,d))

tfidf_ = pd.DataFrame(result, columns = vocab)
tfidf_

,과일이,길고,노란,먹고,바나나,사과,싶은,저는,좋아요
0,0.000000,0.000000,0.000000,0.287682,0.000000,0.693147,0.287682,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.287682,0.287682,0.000000,0.287682,0.000000,0.000000
2,0.000000,0.693147,0.693147,0.000000,0.575364,0.000000,0.000000,0.000000,0.000000
3,0.693147,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.693147,0.693147


# 3. 사이킷런을 이용한 DTM과 TF-IDF 실습

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    'you know I want your love',
    'I like you',
    'what should I do ',    
]

vector = CountVectorizer()

# 코퍼스로부터 각 단어의 빈도수를 기록
print(vector.fit_transform(corpus).toarray())

# 각 단어와 맵핑된 인덱스 출력
print(vector.vocabulary_)

[[0 1 0 1 0 1 0 1 1]
 [0 0 1 0 0 0 0 1 0]
 [1 0 0 0 1 0 1 0 0]]
{'you': 7, 'know': 1, 'want': 5, 'your': 8, 'love': 3, 'like': 2, 'what': 6, 'should': 4, 'do': 0}


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    'you know I want your love',
    'I like you',
    'what should I do ',    
]

tfidfv = TfidfVectorizer().fit(corpus)
print(tfidfv.transform(corpus).toarray())
print(tfidfv.vocabulary_)

[[0.         0.46735098 0.         0.46735098 0.         0.46735098
  0.         0.35543247 0.46735098]
 [0.         0.         0.79596054 0.         0.         0.
  0.         0.60534851 0.        ]
 [0.57735027 0.         0.         0.         0.57735027 0.
  0.57735027 0.         0.        ]]
{'you': 7, 'know': 1, 'want': 5, 'your': 8, 'love': 3, 'like': 2, 'what': 6, 'should': 4, 'do': 0}
